In [ ]:
# msd_circuit function
noncliff_prefix, main_cliff_circ = magic_state_dist_steane()
tomo_circs = single_qubit_state_tomography()

# NOTE: maybe do some postprocessing in-between here

# NOTE: unfortunately, I think I need to pass in "target_bloch" here too
exp = PostSelectionExperiment(noncliff_prefix, main_cliff_circ, tomo_circ, postselection_condition, decoder_postselection, decoder_final, decoder_init_args)
# ^ for implementability, I'm going to separate this out.
# decoder_final -> constructor that takes in a DEM and outputs a BaseDecoder (can be used for both the ancillae and output qubit decoders)
# decoder_postselection -> function that takes in the "actual" MSD circuit + trained decoders for each basis and outputs a ConfidenceDecoder
# ^ for MLD, does the sampling shabang
# ^ for MLE, simply just gives back the BaseDecoders for the ancillae (as they are already ConfidenceDecoders)

#PostSelectionExperimentCache a field of PostSelectionExperiment
# - dem_circuits
# - dems
# - decoders --> did we want this field? I think it is covered by the two below
# - initialized_decoder_postselecton --> going to change the format of this field based on what I currently have implemented (for ease of implementation)
# - initializaed_decoder_final
# - raw_results
# - decoded_results


# So there are two ways of going about this:
# 1. Have the prep_decoder, get_samples functions call the other functions internally in their implementations
# 2. The user HAS to call exp.kernels(), exp.dem_circuits(param), exp.dems(), and exp.initialize_decoders() EXPLICITLY.

# some introspection tools that can be skipped/wrapped in a final run_experiment call
# Q: what is the difference between exp.prep_decoder() and exp.initialize_decoders()???
# Maybe exp.prep_decoder() can specifically contain the logic for creating the ConfidenceDecoder ?
# the implementation of exp.prep_decoder()
# ^ TBH I don't know what prep_decoder does. I think we just need initialize_decoders()

decoders = exp.initiliaze_decoders() # in decoder_init_args we might provide a Device Simulator object for decoders that need a simulation
# Q: if decoders only take in DEM's and not the circuit, then how do they simulate atom loss? Can they still accurately simulate atom loss?
# depends on
#dems = exp.dems()
# which depends on
#dem_circuits = exp.dem_circuits(param)
# which depends on exp.kernels which 


# TODO: continue here, 6/8 4:43 PM

raw_results = exp.get_samples(actual_hardware_device)
# returns an array samples x circuits x detectors + postselection observables + observables
# NOTE: for exp.make_task, that is implicitly getting called in prep_decoder(), because we need to construct them there anyways
# depends on exp.make_task
# depends on exp.kernels

decoded_results = exp.decode_and_postselect() # main thing to discuss at next meeting
# returns an array samples x circuits x observables and an array of confidence scores, have them sorted descending order
# ^ NOTE: the shape of the numpy arrays across bases could be different.
# depends on exp.decode_postselection and on exp.decode_final which both are supposed to return (correction, confidence_score), just set confidence_score = 1.0 if decoder does not support that
# ^ not sure if we wanted to implement the raw shot counts for decode_postselection and decode_final. tbh, I could, but I don't really expose it in maybe the shot format that is expected (I store it in a more "condensed" form)
# ^ NOTE: Not quite. decode_final does NOT return confidence_score. Only decode_postselection returns confidence_score

# I don't think this is method should be exposed to the user. It's not like we can specify any continuous accepted_fraction that we want. It's more like
# we have pointwise thresholds that are computed 
tomo_result = exp.tomography_result(accepted_fraction,method)
# I see the benefit of decoupling the tomography result from the fidelity computation. However, the tomography result isn't really ever exposed to the user in our implementation..
# And, if the tomography result isn't ever exposed to the user, then it shouldn't matter (for now) how we compute fidelity on the tomography result.. for now, we just compute fidelity in one
# wrapper function w/o a method on tomo_result.
f = tomo_result.fidelity_bloch(theta, phi)


dataset = exp.analysis_f_vs_fraction()
fig, ax = exp.analysis_visualization()

# combine the figures -- on their own.

# one notebook with this another one with run everything.

# Doc guidelines --> this is a wrapper for the low-level stateless functions in AGENTS.md.
# lookup tabledecoder by syndrome weight.
# ^ change the implementation. -- we can talk about this. how much loss in fidelity would we get?

# think about the tabledecoder 3 bases for fidelity. -- maybe talk to Chen? And think about the API.

# Q: what if you DON'T want to use tsim ???? (and use clifft instead, for example)
# ^ Q: so we don't like this notion of a "task" ? doesn't seem like it

# Q: detector patterns on your ancillae and constructing the ranking table? -- confused on how exactly they do sliding-scale postselection here or WHY they do it this way; think about it later!

In [6]:
from bloqade import squin
from kirin.dialects import ilist

In [7]:
@squin.kernel
def msd_forward(reg):
    squin.broadcast.sqrt_x(ilist.IList([reg[0], reg[1], reg[4]]))
    squin.broadcast.cz(ilist.IList([reg[0], reg[2]]), ilist.IList([reg[1], reg[3]]))
    squin.broadcast.sqrt_y(ilist.IList([reg[0], reg[3]]))
    squin.broadcast.cz(ilist.IList([reg[0], reg[3]]), ilist.IList([reg[2], reg[4]]))
    squin.sqrt_x_adj(reg[0])
    squin.broadcast.cz(ilist.IList([reg[0], reg[1]]), ilist.IList([reg[4], reg[3]]))
    squin.broadcast.sqrt_x_adj(reg)

In [8]:
msd_forward.print()

func.func @msd_forward(reg : !Any) -> !py.NoneType {
  ^0(%msd_forward_self, %reg):
  │  %0 = py.constant.constant 0 : !py.int
  │  %1 = py.indexing.getitem(%reg, %0) : !Any
  │  %2 = py.constant.constant 1 : !py.int
  │  %3 = py.indexing.getitem(%reg, %2) : !Any
  │  %4 = py.constant.constant 4 : !py.int
  │  %5 = py.indexing.getitem(%reg, %4) : !Any
  │  %6 = py.ilist.new(values=(%1, %3, %5)){elem_type=!Any} : !py.IList[!Any, Literal(3,int)]
  │  %7 = func.invoke sqrt_x(%6) : !py.NoneType maybe_pure=False
  │  %8 = py.constant.constant 0 : !py.int
  │  %9 = py.indexing.getitem(%reg, %8) : !Any
  │ %10 = py.constant.constant 2 : !py.int
  │ %11 = py.indexing.getitem(%reg, %10) : !Any
  │ %12 = py.ilist.new(values=(%9, %11)){elem_type=!Any} : !py.IList[!Any, Literal(2,int)]
  │ %13 = py.constant.constant 1 : !py.int
  │ %14 = py.indexing.getitem(%reg, %13) : !Any
  │ %15 = py.constant.constant 3 : !py.int
  │ %16 = py.indexing.getitem(%reg, %15) : !Any
  │ %17 = py.ilist.new(values=(%1